# chesszero — train on Colab (GPU)

Run the AlphaZero-style self-play + training loop on a Colab GPU, saving
checkpoints to Google Drive so they survive disconnects.

**Before you start:** set the runtime to a GPU via **Runtime → Change runtime
type → Hardware accelerator → GPU** (a free T4 is fine).

**How the loop persists work:** checkpoints and sampled games are written under
`<dir>/<git-commit-hash>/`, and every run records a `config.json` with the exact
hyperparameters. Training refuses to start on a dirty git tree, so the hash always
identifies reproducible code — that's why we clone a clean repo below rather than
editing files in the notebook.

## 1. Check the GPU

If `nvidia-smi` errors or CUDA is `False`, fix the runtime type (above) before
continuing — self-play on CPU here is slow.

In [ ]:
!nvidia-smi || echo 'No GPU — set Runtime > Change runtime type > GPU'
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2. Clone the repo

The `batch-self-play` branch has GPU-batched self-play (`--parallel-games`) and the
`--checkpoint-dir` flag used below. Push it to the remote first if you haven't
(`git push -u origin batch-self-play`). For a **private** repo, replace the URL
with a token form: `https://<TOKEN>@github.com/albrodfer1/chess.git`.

In [ ]:
REPO_URL = 'https://github.com/albrodfer1/chess.git'
BRANCH = 'batch-self-play'

import os
if not os.path.isdir('/content/chess'):
    !git clone --branch $BRANCH $REPO_URL /content/chess
else:
    !cd /content/chess && git fetch origin && git checkout $BRANCH && git pull --ff-only
%cd /content/chess
!git log --oneline -1

## 3. Install the package

Colab already ships a CUDA build of PyTorch, so we install the package **without
its dependencies** (`--no-deps`) to avoid pip swapping in a CPU/incompatible torch
and breaking the GPU. We add only `python-chess` (NumPy is already present).

In [ ]:
!pip install -q python-chess
!pip install -q -e . --no-deps
# Sanity check: the CLI and package import, and torch still sees CUDA.
import torch; print('CUDA still available:', torch.cuda.is_available())
!chesszero --help >/dev/null && echo 'chesszero CLI OK'

## 4. Mount Google Drive

Checkpoints and sampled games will be written here so they persist across Colab
disconnects. You'll be asked to authorize access to your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/chesszero')
CKPT_DIR = DRIVE_ROOT / 'checkpoints'
GAMES_DIR = DRIVE_ROOT / 'games'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
GAMES_DIR.mkdir(parents=True, exist_ok=True)
print('Saving checkpoints to:', CKPT_DIR)
print('Saving sampled games to:', GAMES_DIR)

## 5. Train

Key flags:

- `--device cuda` — use the GPU.
- `--parallel-games` — self-play games run **concurrently**, batching their MCTS
  evaluations into single forward passes. This is what actually uses the GPU;
  raise it (32–64+) until the device is saturated or you hit an out-of-memory
  error. Watch utilization with `!nvidia-smi` in another cell.
- `--simulations` — MCTS sims per move (strength vs. speed).
- `--checkpoint-dir` / `--games-dir` — point persistence at Drive.
- `--sample-games` — how many full games to save (for the browser viewer),
  spread evenly across the run.

Checkpoints land in `CKPT_DIR/<git-hash>/model_iter_*.pt` (plus `latest.pt` and
`config.json`). Output streams live; games log in **finish** order (they run
concurrently), so the `game k/N` label is the start index.

In [ ]:
!chesszero --device cuda loop \
    --iterations 20 \
    --games 40 \
    --parallel-games 32 \
    --simulations 200 \
    --sample-games 10 \
    --checkpoint-dir "$CKPT_DIR" \
    --games-dir "$GAMES_DIR"

## 6. Resume after a disconnect

Colab sessions time out (and free GPUs get reclaimed) well before a long run
finishes. Re-run cells 1–4, then this cell: it picks up from `latest.pt` on Drive
and continues.

> Note: `--resume` restores the network and optimizer but **not** the replay
> buffer (it starts empty and refills), and the git hash must match the code you
> resume with — so resume on the same branch/commit.

In [ ]:
import glob
latest = sorted(glob.glob(str(CKPT_DIR / '*' / 'latest.pt')))
assert latest, f'No checkpoint found under {CKPT_DIR} — run cell 5 first.'
RESUME = latest[-1]
print('Resuming from:', RESUME)

!chesszero --device cuda loop \
    --iterations 20 \
    --games 40 \
    --parallel-games 32 \
    --simulations 200 \
    --sample-games 10 \
    --resume "$RESUME" \
    --checkpoint-dir "$CKPT_DIR" \
    --games-dir "$GAMES_DIR"

## 7. Inspect what's on Drive

List saved checkpoints and the config each run used.

In [ ]:
import json, glob
for cfg in sorted(glob.glob(str(CKPT_DIR / '*' / 'config.json'))):
    run = Path(cfg).parent
    iters = len(glob.glob(str(run / 'model_iter_*.pt')))
    c = json.load(open(cfg))
    print(f"{run.name}: {iters} checkpoints | sims={c['num_simulations']} "
          f"games/iter={c['games_per_iteration']} parallel={c.get('selfplay_batch_size')} "
          f"net={c['num_res_blocks']}x{c['num_filters']}")

## 8. (Optional) Evaluate two checkpoints against each other

Play a later model vs. an earlier one to check for real progress (many draws
between similar-strength models is expected — see the course's debugging chapter).

In [ ]:
# Edit these to two checkpoints you actually have (see cell 7):
MODEL_A = str(CKPT_DIR / '<git-hash>' / 'model_iter_20.pt')
MODEL_B = str(CKPT_DIR / '<git-hash>' / 'model_iter_1.pt')

!chesszero --device cuda eval "$MODEL_A" --model-b "$MODEL_B" --games 10 --simulations 200

## Notes

- **Viewer:** the browser game viewer isn't practical inside Colab. Download the
  saved game JSON from `GAMES_DIR` and run `chesszero viewer --games-dir <dir>`
  locally.
- **Fresh vs. bigger network:** the defaults (`6x128`) are laptop-sized. On a GPU,
  a bigger tower (e.g. `num_res_blocks=10`, `num_filters=256`) is stronger and uses
  the hardware better — but those aren't loop flags; set them in `config.py` (or add
  flags) and note that changing the input representation invalidates old
  checkpoints.
- **Keeping the session alive:** long runs may still be cut short; lean on cell 6
  (resume) and check back periodically.